# Extract trace stack from 05010

This notebook extracts a grid of `x1d` traces from the 05010 FLT file, collates them into a `TRACES` BinTableHDU, and saves a new x1d with trace stack + mask.

In [ ]:
%matplotlib widget

import os
import sys
from pathlib import Path

import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt

sys.path.append("../")
import extraction_utils as extract_utils

# Optional: set CRDS paths if they are not already configured.
# Update CRDS_PATH to match your local cache location if needed.
CRDS_PATH = "/Users/parke/crds_cache"
if "CRDS_PATH" not in os.environ:
    os.environ["CRDS_PATH"] = CRDS_PATH
    os.environ.setdefault("CRDS_SERVER_URL", "https://hst-crds.stsci.edu")
    os.environ.setdefault("iref", f"{CRDS_PATH}/references/hst/iref/")
    os.environ.setdefault("jref", f"{CRDS_PATH}/references/hst/jref/")
    os.environ.setdefault("oref", f"{CRDS_PATH}/references/hst/oref/")
    os.environ.setdefault("lref", f"{CRDS_PATH}/references/hst/lref/")
    os.environ.setdefault("nref", f"{CRDS_PATH}/references/hst/nref/")
    os.environ.setdefault("uref", f"{CRDS_PATH}/references/hst/uref/")

In [ ]:
TEST_DATA_DIR = extract_utils.find_test_data_dir()
TARGET_FILE = TEST_DATA_DIR / "of9b05010_flt.fits"
OUTPUT_X1D = TARGET_FILE.with_name(TARGET_FILE.name.replace("_flt", "_x1d_traces"))

X1D_PARAMS = extract_utils.get_x1d_trace_params(extract_utils.default_x1d_params)
STEP = float(X1D_PARAMS["extrsize"])  # pixels

result = extract_utils.build_trace_stack_x1d(
    fltfile=TARGET_FILE,
    output_x1d=OUTPUT_X1D,
    step=STEP,
    x1d_params=X1D_PARAMS,
    spectrum_column="flux",
)

print(f"Wrote: {result['output_x1d']}")
print(f"Traces: {len(result['y_positions'])}")

In [ ]:
# Plot extraction locations and widths on the FLT image.
data = fits.getdata(TARGET_FILE, 1)
ny, nx = data.shape
extrsize = float(X1D_PARAMS["extrsize"])

fig, ax = plt.subplots()
ax.set_title("Extraction locations")
ax.imshow(np.cbrt(data), aspect="auto")

colors = plt.cm.viridis(np.linspace(0, 1, len(result["y_positions"])))
for y, color in zip(result["y_positions"], colors):
    ax.axhspan(y - extrsize / 2, y + extrsize / 2, color=color, alpha=0.15)
    ax.plot([0, nx - 1], [y, y], color=color, lw=0.6)

ax.set_xlim(0, nx - 1)
ax.set_ylim(ny - 1, 0)

In [ ]:
# Plot stacked trace spectra with colors by extraction location.
with fits.open(OUTPUT_X1D) as hdul:
    trace_table = hdul["TRACES"].data
    traces = extract_utils._get_column(trace_table, "flux")
    wavelength = extract_utils._get_column(trace_table, "wavelength")

if traces is None or wavelength is None:
    raise ValueError("Missing FLUX/WAVELENGTH columns in TRACES table.")

colors = plt.cm.viridis(np.linspace(0, 1, traces.shape[0]))
fig, ax = plt.subplots()
for spectrum, wave, color in zip(traces, wavelength, colors):
    ax.plot(wave, spectrum, color=color, alpha=0.8, lw=0.7)
ax.set_title("Trace spectra")
ax.set_xlabel("Wavelength")
ax.set_ylabel("Flux")